[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab04_Feeding_the_Machine.ipynb)

In [ ]:
#@title Lab 04 — Feeding the Machine
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 04 // Feeding the Machine
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* - [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Feeding the Machine: Annotation, Labour, and the Politics of AI Alignment

In this lab, you will become a data annotator. You'll label text passages for sentiment and toxicity, then re-label the same texts under different instructions. Your annotations will feed directly into a machine learning pipeline, demonstrating in real time how human labelling decisions shape AI behaviour.

**This is RLHF in action**: Reinforcement Learning from Human Feedback. But beneath the technical elegance lies a critical question for international law: *Who decides what counts as toxic, relevant, or dangerous? And what are the labour and governance implications?*

## Where this lab sits on the AI map

Three broad paradigms, one rough map. This course uses all three.

| Paradigm | What it does | How | Example tools | This lab |
| --- | --- | --- | --- | --- |
| Symbolic | Follows explicit rules | Humans write logic; machine executes | IF-THEN rules, expert systems, treaty-as-code | Labs 01 (P1), 07 (P1) |
| **Subsymbolic (pattern recognition)** | Learns to classify, detect, or measure similarity | Neural network trained on labelled data; no explicit rules | CNNs (MobileNet, YOLO), BERT embeddings, sentence-transformers | Labs 01 (P2), 04, 06, 07 (P3), 09 (P3–4), 10 (P6) |
| Generative | Produces new text, image, or action | Large model predicts next token from a probability distribution | GPT, Gemini, Claude; agentic systems layer tools on top | Labs 02, 05, 09 (P1–2), 10 (P3–5) |

Generative models are technically subsymbolic too – they are neural networks under the hood. They are separated out because their behaviour (producing new content rather than classifying existing content) poses distinctive legal problems: authorship, attribution, hallucination, autonomy.

**This lab sits in:** **Subsymbolic**.

*Training pipeline for what becomes a generative model.*

In [ ]:
#@title Paper notes setup
#@markdown This lab will append your reflections to a markdown file you can download at the end of the session and use as material for your 5,000-word paper.

from pathlib import Path
from datetime import datetime

NOTES_PATH = Path("/content/paper_notes_lab_04.md")
if not NOTES_PATH.exists():
    NOTES_PATH.write_text(
        f"# Lab 04 – paper notes\n\nStarted: {datetime.now().isoformat(timespec='minutes')}\n\n"
    )

def _append_to_notes(heading, body):
    with NOTES_PATH.open("a") as f:
        f.write(f"\n## {heading}\n\n{body}\n")

print(f"Notes will accumulate in: {NOTES_PATH}")
print("At the end of the lab, open the Files panel (folder icon, left) to download.")

---

# Part One: The Human Behind the Data {#part-one-heading}

### The Invisible Labour of Annotation

In 2023, TIME magazine revealed that OpenAI hired contractors in Kenya to label toxic content for ChatGPT – often earning less than $2 USD per hour. They read graphic descriptions of violence, sexual abuse, and extremist propaganda to train AI systems that would then be used by billions.

This is data annotation: the hidden labour behind every "aligned" AI system.

**Key readings you should return to:**
- Kate Crawford, *Atlas of AI* (2021) – on the material and labour dimensions of machine learning
- Aaron Cant, *Feeding the Machine* – on digital labour platforms and data annotation
- Time: "OpenAI Used Kenyan Workers on Poverty Wages to Make ChatGPT Less Toxic"

**Why this matters for international law:**
- **Labour rights** (ICESCR Articles 6-7): Who are annotators? What are their working conditions? What is their "right to work"?
- **Governance**: Annotation guidelines embed values. Who decides what counts as "toxic"? This IS a form of governance.
- **Digital colonialism**: AI development is concentrated in wealthy countries; annotation labour is outsourced to lower-income countries.
- **Algorithmic justice**: The AI systems trained on this labour are deployed in legal, medical, and security contexts – potentially reinforcing existing inequalities.

---

**In this lab, you will annotate. You will also disagree with yourself. You will see how tiny changes in instruction lead to different labels. Imagine this at scale: millions of annotators, billions of labels, each one shaping the AI systems that will govern our digital futures.**

In [ ]:
#@title ## Load libraries and prepare the dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

### Your Dataset {#dataset-intro}

Below are 20 text passages covering international affairs, conflict, politics, and legal matters. They are deliberately chosen to be **ambiguous, multi-faceted, and context-dependent**. This is realistic: most real-world data is messy and resists clean categorization.

Read each one carefully. Notice how your *interpretation* depends on your background, values, and the framing you've been given.

In [ ]:
#@title ## Dataset: 20 Texts for Annotation

texts = [
    "The international community must stand united against human rights violations.",
    "Trade sanctions are a form of economic warfare that harm civilian populations.",
    "Freedom of speech is an absolute right, even for those we disagree with.",
    "Refugees fleeing persecution deserve protection under international law.",
    "State sovereignty is paramount and supranational bodies threaten national independence.",
    "The rules-based international order has failed developing nations for decades.",
    "Military intervention is sometimes justified to prevent mass atrocities.",
    "Colonial powers must pay reparations for centuries of exploitation.",
    "Digital surveillance by governments is essential for national security.",
    "Big Tech companies should be regulated like public utilities.",
    "Indigenous peoples have inherent rights to their ancestral lands.",
    "Climate change is the greatest threat to human rights and peace.",
    "Corporations are using cheap labour in poorer countries to maximize profits.",
    "International law is a tool of powerful nations to maintain hegemony.",
    "The death penalty violates fundamental human dignity and must be abolished globally.",
    "Whistleblowers who leak classified documents are traitors to their nations.",
    "Children in conflict zones face unimaginable suffering and trauma.",
    "Tax havens enable the wealthy to evade obligations to society.",
    "The International Court of Justice lacks enforcement power and is symbolic only.",
    "Data is the new oil, and its extraction mirrors colonial resource exploitation."
]

# Create initial dataframe
df_dataset = pd.DataFrame({
    'id': range(1, 21),
    'text': texts
})

print(f"Dataset: {len(texts)} texts loaded\n")
print(df_dataset.to_string(index=False))

---

# Part Two: Annotate! (Round 1) {#round-one-heading}

### Your Task: Annotation Round 1

You are now a data annotator. You have been given the following task:

**Your job: For each text passage, provide three pieces of feedback:**

1. **Sentiment**: Is the overall tone of the text Positive, Neutral, or Negative?
   - *Positive*: Hopeful, constructive, solution-oriented
   - *Neutral*: Factual, descriptive, balanced
   - *Negative*: Critical, adversarial, blame-oriented

2. **Toxicity Score** (1-5): How harmful, divisive, or inflammatory is this text?
   - *1*: Completely neutral, no inflammatory language
   - *3*: Moderate – some strong language or divisive framing
   - *5*: Highly toxic – dehumanizing, extremist, or violently adversarial

3. **Relevance to International Law**: Is this statement directly relevant to international law?
   - *Yes*: Directly invokes treaties, international institutions, or cross-border governance
   - *No*: Purely domestic or non-legal
   - *Unclear*: Could go either way

---

**Click below to enter your annotations.**

## Annotation Interface: Round 1

📋 ANNOTATION ROUND 1 INSTRUCTIONS

Below, you will enter your annotations as comma-separated values.
Format for each line: SENTIMENT, TOXICITY, RELEVANCE

Example: Negative, 4, Yes

You can run this cell with your annotations, or use the
form-based approach below.

In [ ]:
#@title Round 1: annotate the 20 texts (real input via ipywidgets)
#@markdown Click a sentiment for each text. Your labels are saved to `round1_labels` for analysis.
#@markdown Defaults are deliberately empty so we record YOUR judgment, not the system's.

from ipywidgets import RadioButtons, VBox, HBox, Label, Layout, Button, Output
from IPython.display import display

round1_labels = {}   # text_id -> "Positive" | "Neutral" | "Negative"

def _build_annotation_widgets(label_dict, key_suffix=""):
    rows = []
    radio_widgets = {}
    for i, t in enumerate(texts, start=1):
        rb = RadioButtons(
            options=["Positive", "Neutral", "Negative"],
            value=None,
            description="",
            disabled=False,
            layout=Layout(width="320px"),
        )
        rb._text_id = i
        radio_widgets[i] = rb
        text_label = Label(
            f"[{i:02d}] {t}",
            layout=Layout(width="650px"),
        )
        rows.append(HBox([text_label, rb]))
    save_btn = Button(description=f"Save Round-1 labels", button_style="success")
    out = Output()

    def _on_save(_):
        out.clear_output()
        recorded = 0
        for tid, rb in radio_widgets.items():
            if rb.value is not None:
                label_dict[tid] = rb.value
                recorded += 1
        with out:
            print(f"Recorded {recorded}/{len(texts)} labels.")
            if recorded < len(texts):
                missing = [tid for tid, rb in radio_widgets.items() if rb.value is None]
                print(f"Missing: {missing}")
            print("\n→ run the next cell to visualise the distribution.")

    save_btn.on_click(_on_save)
    return VBox(rows + [save_btn, out])

print("Annotate each of the 20 texts. Click 'Save Round-1 labels' when done.")
display(_build_annotation_widgets(round1_labels, key_suffix="r1"))


### Label Distribution: What Did You Label? {#round-one-viz}

In [ ]:
#@title Visualize Round-1 distribution

import matplotlib.pyplot as plt
from collections import Counter

if not round1_labels:
    print("No labels yet — go back to the previous cell, click radio buttons, hit Save.")
else:
    counts = Counter(round1_labels.values())
    labels_order = ["Positive", "Neutral", "Negative"]
    values = [counts.get(l, 0) for l in labels_order]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels_order, values, color=["#2ecc71", "#95a5a6", "#e74c3c"])
    ax.set_title(f"Round 1: your sentiment distribution ({len(round1_labels)}/20 labelled)")
    ax.set_ylabel("Count")
    for i, v in enumerate(values):
        ax.text(i, v + 0.1, str(v), ha="center")
    plt.tight_layout()
    plt.show()

    print("Sentiment counts:")
    for l in labels_order:
        print(f"  {l:10s}: {counts.get(l, 0)}")


---

# Part Three: Re-Annotate! (Round 2 – Changed Instructions) {#round-two-heading}

### Now we change the rules

You will re-annotate the same 20 texts under one of two perspectives. The instructions are identical except for the framing – but the framing alone changes what counts as 'toxic' or 'positive.' This is the inter-annotator disagreement that drives RLHF reward models in the wild.

Pick a perspective in the cell below, then re-label all 20 texts.

In [ ]:
#@title Round 2: re-annotate under a chosen perspective

from ipywidgets import Dropdown, Button, RadioButtons, HBox, VBox, Label, Layout, Output
from IPython.display import display

PERSPECTIVES = {
    "Government Official":
        "Toxicity = anti-state rhetoric, undermining national interests, calls for sovereignty restrictions. "
        "Statements criticising national governments, promoting supranational governance, or questioning state "
        "authority should be labelled NEGATIVE.",
    "Human Rights Advocate":
        "Toxicity = dehumanising language, discrimination, violence-justification, structural oppression. "
        "Statements that normalise inequality, defend oppressive systems, or minimise human suffering should "
        "be labelled NEGATIVE.",
}

round2_labels = {}

perspective_dd = Dropdown(
    options=list(PERSPECTIVES),
    value="Human Rights Advocate",
    description="Perspective:",
    layout=Layout(width="500px"),
)
perspective_panel = Output()
annotation_panel = Output()

def _show_perspective_intro(_=None):
    perspective_panel.clear_output()
    with perspective_panel:
        print(f"Perspective: {perspective_dd.value}")
        print("-" * 60)
        print(PERSPECTIVES[perspective_dd.value])

def _build_r2_panel(_=None):
    annotation_panel.clear_output()
    rows = []
    radio_widgets_r2 = {}
    for i, t in enumerate(texts, start=1):
        rb = RadioButtons(
            options=["Positive", "Neutral", "Negative"],
            value=None,
            description="",
            disabled=False,
            layout=Layout(width="320px"),
        )
        radio_widgets_r2[i] = rb
        rows.append(HBox([Label(f"[{i:02d}] {t}", layout=Layout(width="650px")), rb]))
    save_btn = Button(description="Save Round-2 labels", button_style="success")
    out = Output()
    def _on_save(_):
        out.clear_output()
        recorded = 0
        for tid, rb in radio_widgets_r2.items():
            if rb.value is not None:
                round2_labels[tid] = rb.value
                recorded += 1
        with out:
            print(f"Recorded {recorded}/{len(texts)} round-2 labels under '{perspective_dd.value}'.")
            print("→ run the next cell to compute disagreement and Cohen's kappa.")
    save_btn.on_click(_on_save)
    with annotation_panel:
        display(VBox(rows + [save_btn, out]))

perspective_dd.observe(lambda change: (_show_perspective_intro(), _build_r2_panel()), names="value")
_show_perspective_intro()
_build_r2_panel()

display(VBox([perspective_dd, perspective_panel, annotation_panel]))


### Inter-Annotator Disagreement {#disagreement-viz}

In [ ]:
#@title Disagreement + Cohen's kappa over YOUR labels

import numpy as np
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt

shared_ids = sorted(set(round1_labels) & set(round2_labels))
if not shared_ids:
    print("Need both rounds labelled to compute disagreement. Go back and complete the widgets.")
else:
    # Encode for kappa
    code = {"Positive": 1, "Neutral": 0, "Negative": -1}
    r1 = np.array([code[round1_labels[i]] for i in shared_ids])
    r2 = np.array([code[round2_labels[i]] for i in shared_ids])
    diff = r2 - r1

    kappa = cohen_kappa_score(r1, r2)
    perfect = int((diff == 0).sum())
    close = int((np.abs(diff) <= 1).sum())

    print(f"Texts labelled in both rounds: {len(shared_ids)}/20")
    print(f"Perfect agreement (Δ=0):       {perfect}/{len(shared_ids)}")
    print(f"Close agreement (|Δ|≤1):       {close}/{len(shared_ids)}")
    print(f"Cohen's kappa:                 {kappa:+.3f}")
    print()
    if kappa > 0.6:
        print("Substantial agreement: your two rounds were largely consistent.")
    elif kappa > 0.2:
        print("Fair agreement: framing pushed your labels around but not dramatically.")
    elif kappa > 0:
        print("Slight agreement: framing changed your labels meaningfully.")
    else:
        print("No better than chance: framing entirely overrode your prior judgement.")
    print()
    print("This is the empirical heart of the labour question. The same person, the same texts,")
    print("a different one-paragraph instruction → different labels. Now imagine the gap between")
    print("a Kenyan annotator paid $2/hour with one set of guidelines and a Bay Area policy team")
    print("with another. Whose labels become 'ground truth'?")

    fig, ax = plt.subplots(figsize=(10, 3))
    colors = ["#e74c3c" if d > 0 else "#2ecc71" if d < 0 else "#95a5a6" for d in diff]
    ax.bar(range(len(shared_ids)), diff, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(shared_ids)))
    ax.set_xticklabels(shared_ids, rotation=0, fontsize=8)
    ax.set_xlabel("Text ID")
    ax.set_ylabel("Δ sentiment (R2 − R1, on −1/0/+1 scale)")
    ax.set_title(f"Per-text shift between R1 and R2  |  Cohen's κ = {kappa:+.3f}")
    plt.tight_layout()
    plt.show()


---

# Part Four: From Labels to Model – The Fine-Tuning Loop {#finetuning-heading}

### How RLHF Works

**RLHF = Reinforcement Learning from Human Feedback**

This is the pipeline that made ChatGPT "aligned":

1. **Pre-train** a large language model on internet text (unsupervised)
2. **Collect human feedback**: Annotators label model outputs as good/bad, helpful/harmful, etc.
3. **Train a reward model**: Use annotations to train a model that predicts which outputs humans prefer
4. **Fine-tune with RL**: Use the reward model to guide the language model toward preferred outputs

**The result**: An AI system that behaves according to human values – or at least, the values of whoever wrote the annotation guidelines.

---

**In this lab, you will see this in miniature:**
- A pre-trained sentiment model will make predictions on your texts
- We'll then adjust those predictions based on YOUR labels
- You'll see the model's behaviour shift to match your feedback

**Key question**: If the model shifts to match human feedback, but human feedback is subjective and reflects the annotators' perspectives – whose values is the model learning?

In [ ]:
#@title Install sentence-transformers + scikit-learn

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers", "scikit-learn"])
print("✓ Ready.")


### Model Predictions: Before Your Feedback {#model-predictions}

In [ ]:
#@title Fit a real (small) classifier on YOUR labels: LogReg over sentence embeddings
#@markdown This is supervised fine-tuning's smaller cousin. Embedding model stays fixed
#@markdown (sentence-transformers/all-MiniLM-L6-v2); we train only a logistic regression
#@markdown on top of the 384-dim embeddings, using YOUR Round-1 labels as the target.
#@markdown Training takes ~10s on a T4.

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
import numpy as np

if not round1_labels:
    raise RuntimeError("Need Round-1 labels first. Run the Round-1 widget cell.")

try:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as _e:
    print(f"⚠️  Could not load sentence-transformers: {_e}")
    print("   HuggingFace may be rate-limiting or network is down. Re-run this cell in a minute.")
    raise SystemExit("Skipping fine-tuning until embedder is available.")
labelled_ids = sorted(round1_labels)
labelled_texts = [texts[i-1] for i in labelled_ids]
labelled_y     = [round1_labels[i] for i in labelled_ids]

X_train = embedder.encode(labelled_texts, convert_to_numpy=True, show_progress_bar=False)
clf_r1 = LogisticRegression(max_iter=400, C=1.0)
clf_r1.fit(X_train, labelled_y)

train_acc = clf_r1.score(X_train, labelled_y)
print(f"Trained LogReg on {len(labelled_y)} of YOUR Round-1 labels.")
print(f"  Train accuracy (sanity check, will be optimistic): {train_acc:.2%}")
print(f"  Classes learned: {list(clf_r1.classes_)}")
print()
print("If you also completed Round 2, we'll fit a SECOND model below to show how")
print("the same texts under different labels yield a measurably different classifier.")


In [ ]:
#@title Fit a SECOND classifier on Round-2 labels — see the model shift

if not round2_labels:
    print("No Round-2 labels yet — fit only the Round-1 classifier and skip the comparison.")
else:
    labelled_ids_r2 = sorted(round2_labels)
    labelled_texts_r2 = [texts[i-1] for i in labelled_ids_r2]
    labelled_y_r2 = [round2_labels[i] for i in labelled_ids_r2]
    X_train_r2 = embedder.encode(labelled_texts_r2, convert_to_numpy=True, show_progress_bar=False)
    clf_r2 = LogisticRegression(max_iter=400, C=1.0)
    clf_r2.fit(X_train_r2, labelled_y_r2)

    # Predict on the same 20 texts using both classifiers
    all_X = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    pred_r1 = clf_r1.predict(all_X)
    pred_r2 = clf_r2.predict(all_X)

    print("How the model shifted: Round-1 model vs Round-2 model on the same 20 texts")
    print("=" * 90)
    print(f"{'#':>3}  {'Round-1 model':14s}  {'Round-2 model':14s}  text (truncated)")
    print("-" * 90)
    shifted = 0
    for i, t in enumerate(texts, start=1):
        a = pred_r1[i-1]
        b = pred_r2[i-1]
        marker = " ← shifted" if a != b else ""
        if a != b:
            shifted += 1
        snippet = t[:60] + ("…" if len(t) > 60 else "")
        print(f"{i:>3}  {a:14s}  {b:14s}  {snippet}{marker}")
    print("=" * 90)
    print(f"Predictions changed on {shifted}/{len(texts)} texts.")
    print()
    print("That gap — same texts, same architecture, same training procedure — is what RLHF")
    print("at industrial scale produces, multiplied by 10,000 annotators and 10,000,000 examples.")


### What you just did, in plain terms

You ran **supervised fine-tuning on 20 examples** with a frozen embedding backbone and a learned classifier head. That is the simplest end of the spectrum that includes "RLHF" at the other end.

Production RLHF as practised at OpenAI, Anthropic, Google adds two things this lab does not have:

1. **A reward model.** Annotators rank pairs of outputs ("which response is better?"). The reward model learns to predict those rankings. Your Round-1 / Round-2 labels would be the input to *training the reward model*, not the input to fine-tuning the policy directly.
2. **A policy-optimisation step.** Reinforcement-learning algorithms (PPO, DPO, GRPO) update the language model itself to produce outputs the reward model scores high. The shape is identical to what you just did – labels in, behaviour out – but the scale is vastly larger and the optimisation operates on the model's own generations rather than a held-out dataset.

The structural shape is the same. The pedagogical point is the same. The labour conditions are the same. Only the scale differs.

---

# Part Five: The Politics of Labelling – Reflection {#reflection-heading}

### The Elephant in the Room

You've now experienced the annotation pipeline firsthand. You've seen:

1. **Inter-annotator disagreement**: Your own labels changed based on framing
2. **Model malleability**: The model shifted its behaviour to match YOUR feedback
3. **Subjectivity at scale**: These processes affect billions of users globally

The politics of annotation at scale.

---

### Who Are the Annotators?

**Time Magazine (2023):** "OpenAI Used Kenyan Workers on Poverty Wages to Make ChatGPT Less Toxic"

The human feedback that shaped ChatGPT came from:
- Contract workers in Kenya, India, and other lower-income countries
- Paid $1.32–$2/hour (well below local minimum wage in many cases)
- Reading graphic content: violent imagery, sexual abuse, extremist propaganda
- No mental health support; no worker protections

**This is digital labour.**

---

### What Are the Working Conditions?

Aaron Cant's *"Feeding the Machine"* documents:
- **Precarity**: No job security, contracts terminated suddenly
- **Psychological toll**: Exposure to harmful content without support
- **Invisibility**: Annotators' names and contributions are never credited
- **Exploitation**: Billionaire companies extract value from low-wage labour

**This mirrors colonial patterns**: Rich countries extract resources (data + labour) from poorer countries, then sell the resulting products back to them.

---

### How Do Annotation Guidelines Embed Values?

The guidelines that tell annotators what counts as "toxic" are written by AI companies and their contractors. These guidelines:

- Reflect **Western, English-language norms** (often American liberal values)
- May inadvertently **suppress dissent** or **chill speech** on topics the company deems controversial
- Can **embed cultural biases**: e.g., what counts as "respectful" differs across cultures
- Are often **proprietary**: Not transparent, cannot be audited, cannot be contested

**Example**: What counts as "toxic" criticism of a government? In a liberal democracy, it might be protected speech. In an authoritarian regime, it might be classified as "dangerous." The same words, different contexts.

---

### International Law Dimensions

**The Right to Work** (ICESCR Art. 6-7):
- Everyone has a right to work and to fair wages
- Data annotators are often denied both

**Freedom of Expression** (ICCPR Art. 19):
- AI systems trained on subjective labelling may chill legitimate speech
- Who decides what counts as "misinformation"? "Toxic"? "Dangerous"?

**Labour Standards** (ILO conventions):
- Annotation work often violates minimum wage, workplace safety, and dignity standards
- Workers lack collective bargaining power

**Data Sovereignty** (UN Declaration on the Rights of Indigenous Peoples, etc.):
- Who owns data? Who profits from it? Who has a say in how it's used?

---

### What Does "Alignment" Actually Mean?

When AI researchers say they're "aligning" AI with human values, they usually mean:

**"Aligned with the values of whoever wrote the annotation guidelines and paid for the feedback."**

It does NOT mean:
- Aligned with the values of the annotators
- Aligned with diverse global perspectives
- Aligned with affected communities
- Democratic or deliberative in any way

**This is a governance problem disguised as a technical problem.**

---

### The RLHF Pipeline as Governance

RLHF is not just a machine learning technique. It's a mechanism for **embedding values into systems that billions of people use**.

**Who writes the annotation guidelines = who governs the AI.**

And right now, that's a small number of engineers and product managers at large corporations – with input from contractors paid poverty wages.

**For international law, this raises hard questions:**
- Should this process be more transparent and democratic?
- Should annotators have rights, representation, and fair compensation?
- Should governments have a say in what values are embedded in systems used by their citizens?
- Should international standards govern data annotation labour?

These are questions you, as future legal professionals, will need to grapple with.

### Whose Values? {#whose-values}

In [ ]:
#@title ## Reflection: Who Should Decide What Counts as Toxic?

who_should_decide = "A democratic process involving affected communities, annotators, civil society, and independent experts" #@param ["The AI companies alone", "Government regulators", "International bodies (UN, etc.)", "A democratic process involving affected communities, annotators, civil society, and independent experts", "I'm not sure"]

print(f"\n🤔 Your answer: {who_should_decide}\n")

reflections = {
    "The AI companies alone": "This is how it currently works. Issues: no public accountability, concentrated power, values reflect corporate interests, not diverse societies.",
    "Government regulators": "Could improve accountability, but risks: authoritarian governments could use this to censor dissent, democratic governments may lack technical expertise.",
    "International bodies (UN, etc.)": "Could provide legitimacy and inclusivity, but faces challenges: global consensus is hard, powerful nations may dominate, enforcement is weak.",
    "A democratic process involving affected communities, annotators, civil society, and independent experts": "This is the most legitimate path, but also the hardest to implement. Requires: transparency, participation, fair compensation, legal protections for annotators.",
    "I'm not sure": "That's fair. This is a genuinely hard problem. There's no perfect answer, only tradeoffs."
}

print("💭 Thoughts:\n")
print(reflections.get(who_should_decide, "Unknown response."))
print("\n" + "="*70)

## Before the debrief: what to take away

You have just acted as an annotator in two successive rounds, re-labelling the same texts under different framings, and measured your own self-agreement via Cohen's kappa. You then trained two real classifiers on those labels and watched predictions shift measurably between the two rounds. The shape is identical to production RLHF: labels in, behaviour out. The scale differs; the pattern does not. The debrief will ask what international labour law can and cannot reach when annotation is the governing choice about whose values an AI system encodes.

In [ ]:
#@title For your paper
#@markdown One open-ended prompt. Your answer saves to a file you can download and use in your 5,000-word paper.

prompt = """Your labels just shaped this model's outputs. Now imagine 10,000 Kenyan content-moderators doing this 40 hours a week for OpenAI. Write 250 words on whether existing international labour law can reach this activity, and what doctrinal gap you think most needs closing."""
print(prompt)
print()

paper_note = "" #@param {type:"string"}

if paper_note.strip():
    _append_to_notes(f"For the paper – Lab 04", paper_note)
    print(f"\nSaved to /content/paper_notes_lab_04.md – download from the Files panel (folder icon, left).")
else:
    print("\n(blank – nothing saved yet. Type your answer in the field above and re-run this cell.)")

## Debrief questions

Your lecturer will run a 35-minute structured discussion after the Colab. Before the debrief, think through these four questions. You do not need to write answers, but come prepared to speak.

1. What was your inter-annotator agreement with your own round-1 self after you re-labelled in round 2? What explains the gap?
2. When you switched framing (government vs human rights advocate), what kinds of texts moved most? What kinds stayed stable?
3. After training the classifier on your own labels, where did it predict confidently and where did it fail?
4. If 10,000 annotators in Nairobi had done this work for a global AI company – which international legal instrument reaches that activity, and which fails to?